Question 4 - Augmentation

In [ ]:
import Modified_code as orig

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

device = orig.device

In [ ]:

# DEFINE INDIVIDUAL AUGMENTATIONS AND DEMONSTRATION

# Download a raw copy of CIFAR-10 WITHOUT normalization for visual plot clarity
visualization_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms.ToTensor())
raw_image, label_idx = visualization_dataset[4]  # For Autombile
# raw_image, label_idx = visualization_dataset[29]  # For Airplane 
class_names = visualization_dataset.classes

# Define standalone transformation policies
crop_transform = transforms.RandomCrop(size=32, padding=4)
rotation_transform = transforms.RandomRotation(degrees=15)
flip_transform = transforms.RandomHorizontalFlip(p=1.0)
zoom_transform = transforms.Compose([
    transforms.Resize(40),
    transforms.CenterCrop(32)
])

to_pil = transforms.ToPILImage()
pil_img = to_pil(raw_image)

transformed_images = [
    raw_image,
    transforms.ToTensor()(crop_transform(pil_img)),
    transforms.ToTensor()(rotation_transform(pil_img)),
    transforms.ToTensor()(zoom_transform(pil_img)),
    transforms.ToTensor()(flip_transform(pil_img))
]

titles = ["Original", "Random Crop", "Random Rotation", "Zoom Effect", "Horizontal Flip"]

# Generate comparative plot layout
fig, axes = plt.subplots(1, 5, figsize=(15, 4))
fig.suptitle(f"Figure 15: Visual Analysis of Data Augmentations on Class [{class_names[label_idx]}]", fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    img_display = transformed_images[i].permute(1, 2, 0).numpy()
    ax.imshow(img_display)
    ax.set_title(titles[i], fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:

#  FILTERING DATASET TO 2 TARGET CLASSES (SUBSET)

# Train transforms with integrated regularizations
augmented_train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Evaluation transforms (Clean validation tracking)
clean_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

full_train_dataset = datasets.CIFAR10(root='./data', train=True, download=True)
full_test_dataset = datasets.CIFAR10(root='./data', train=False, download=True)

# Select target tracking indexes (0: Airplane, 1: Automobile)
TARGET_CLASSES = [0, 1] 

train_indices = [i for i, label in enumerate(full_train_dataset.targets) if label in TARGET_CLASSES]
test_indices = [i for i, label in enumerate(full_test_dataset.targets) if label in TARGET_CLASSES]

class SubsetWithTransform(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
        
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        y = TARGET_CLASSES.index(y)
        return x, y
        
    def __len__(self):
        return len(self.subset)

raw_train_subset = torch.utils.data.Subset(datasets.CIFAR10(root='./data', train=True, transform=None), train_indices)
raw_test_subset = torch.utils.data.Subset(datasets.CIFAR10(root='./data', train=False, transform=None), test_indices)

augmented_train_dataset = SubsetWithTransform(raw_train_subset, augmented_train_transform)
clean_test_dataset = SubsetWithTransform(raw_test_subset, clean_test_transform)

batch_size = 64 
train_loader_aug = DataLoader(augmented_train_dataset, batch_size=batch_size, shuffle=True)
test_loader_clean = DataLoader(clean_test_dataset, batch_size=batch_size, shuffle=False)

print(f"[Pipeline Status] Filtered dataset successfully to {len(TARGET_CLASSES)} target classes.")
print(f"Optimized Training Subset Size: {len(augmented_train_dataset)} samples")
print(f"Optimized Testing Subset Size: {len(clean_test_dataset)} samples")

In [ ]:

# INITIALIZING AND TRAINING USING THE IMPORTED MODULE

# Pull the CNN class definition directly from your script
model_augmented = orig.InitialCNN().to(device)

# Modify output layers to prevent dimensional errors during binary loss passes
model_augmented.fc2 = nn.Linear(256, 2).to(device)

criterion = nn.CrossEntropyLoss()
optimizer_aug = optim.Adam(model_augmented.parameters(), lr=0.001)

num_epochs_aug = 15
aug_train_losses, aug_test_losses = [], []
aug_train_accs, aug_test_accs = [], []

print("Executing CNN Retraining Loop using Augmented Subsets (2 Classes)...")
print("-" * 80)

for epoch in range(num_epochs_aug):
    # CRITICAL: Dynamically direct the imported script's global pointer to look at our local optimizer instance
    orig.optimizer = optimizer_aug
    
    # Use the imported modular training/testing functions
    train_loss, train_acc, _ = orig.train(model_augmented, train_loader_aug)
    test_loss, test_acc = orig.test(model_augmented, test_loader_clean)
    
    aug_train_losses.append(train_loss)
    aug_test_losses.append(test_loss)
    aug_train_accs.append(train_acc)
    aug_test_accs.append(test_acc)
    
    print(f"Epoch [{epoch+1:02d}/{num_epochs_aug}] -> "
          f"Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f} | "
          f"Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

print("-" * 80)
print("Augmented subset CNN training phase complete.")

In [ ]:
#  PLOTTING PERFORMANCE EVALUATION SYMMETRIC CURVES

epochs_aug_range = range(1, num_epochs_aug + 1)

# FIXED: We extract the baseline training history arrays directly from the imported script module
baseline_epochs = range(1, len(orig.train_losses) + 1) 

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- PANEL 1: LOSS CURVES COMPARISON ---
axes[0, 0].plot(baseline_epochs, orig.train_losses, label='Train Loss (Baseline 10-Class)', color='#1f77b4', linestyle='--')
axes[0, 0].plot(baseline_epochs, orig.test_losses, label='Test Loss (Baseline 10-Class)', color='#ff7f0e', linestyle='--')
axes[0, 0].set_title('Figure 16(a): Baseline CNN Loss History', fontsize=11, fontweight='bold')
axes[0, 0].set_xlabel('Epochs')
axes[0, 0].set_ylabel('Loss Value')
axes[0, 0].grid(True, linestyle=':', alpha=0.5)
axes[0, 0].legend()

axes[0, 1].plot(epochs_aug_range, aug_train_losses, label='Train Loss (Augmented 2-Class)', color='#1f77b4', linewidth=2)
axes[0, 1].plot(epochs_aug_range, aug_test_losses, label='Test Loss (Augmented 2-Class)', color='#ff7f0e', linewidth=2)
axes[0, 1].set_title('Figure 16(b): Augmented Subset CNN Loss History', fontsize=11, fontweight='bold')
axes[0, 1].set_xlabel('Epochs')
axes[0, 1].set_ylabel('Loss Value')
axes[0, 1].grid(True, linestyle=':', alpha=0.5)
axes[0, 1].legend()

# --- PANEL 2: ACCURACY CURVES COMPARISON ---
axes[1, 0].plot(baseline_epochs, orig.train_accs, label='Train Acc (Baseline 10-Class)', color='#2ca02c', linestyle='--')
axes[1, 0].plot(baseline_epochs, orig.test_accs, label='Test Acc (Baseline 10-Class)', color='#d62728', linestyle='--')
axes[1, 0].set_title('Figure 17(a): Baseline CNN Accuracy Trends', fontsize=11, fontweight='bold')
axes[1, 0].set_xlabel('Epochs')
axes[1, 0].set_ylabel('Accuracy (%)')
axes[1, 0].grid(True, linestyle=':', alpha=0.5)
axes[1, 0].legend()

axes[1, 1].plot(epochs_aug_range, aug_train_accs, label='Train Acc (Augmented 2-Class)', color='#2ca02c', linewidth=2)
axes[1, 1].plot(epochs_aug_range, aug_test_accs, label='Test Acc (Augmented 2-Class)', color='#d62728', linewidth=2)
axes[1, 1].set_title('Figure 17(b): Augmented Subset CNN Accuracy Trends', fontsize=11, fontweight='bold')
axes[1, 1].set_xlabel('Epochs')
axes[1, 1].set_ylabel('Accuracy (%)')
axes[1, 1].grid(True, linestyle=':', alpha=0.5)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# SUBSET CONFUSION MATRIX VISUALIZATION
from sklearn.metrics import confusion_matrix
import seaborn as sns

def plot_subset_confusion_matrix(model_instance, loader, compute_device, plot_title):
    model_instance.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(compute_device)
            outputs = model_instance(images)
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    filtered_class_names = ['airplane', 'automobile']
    raw_cm = confusion_matrix(all_labels, all_preds)
    cm_percent = raw_cm.astype('float') / raw_cm.sum(axis=1)[:, np.newaxis] * 100

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_percent, annot=True, fmt=".1f", cmap="Blues",
                xticklabels=filtered_class_names, yticklabels=filtered_class_names)
    
    plt.title(plot_title, fontsize=13, fontweight='bold')
    plt.ylabel('True Label (Ground Truth)', fontsize=11)
    plt.xlabel('Predicted Label (Model Inference)', fontsize=11)
    plt.tight_layout()
    plt.show()

# Render evaluation output
plot_subset_confusion_matrix(
    model_augmented, 
    test_loader_clean, 
    device, 
    plot_title="Figure 18: Augmented Binary CNN Normalized Confusion Matrix (%)"
)